# Phase 0 -- Feasibility

Goal: confirm the reuse thesis in the LARMOR design doc *before* building any UI.

1. **Reproduce an existing dmfit fit** (`CaAlGlass.fxmla`) using mrsimulator + lmfit, starting from
   the exact parameters dmfit already found, and check the simulated lineshape against dmfit's own
   embedded experimental spectrum.
2. **Open one real Bruker TopSpin EXPNO** from the July 2026 dataset with `nmrglue`, strictly
   read-only, and cross-check the raw FID against TopSpin's own already-processed spectrum.

No data is copied into this repo. Both source paths below are read directly from their original
locations and opened read-only.


In [ ]:
from pathlib import Path

# Read-only sources -- never written to. Forward slashes work fine with Path() on Windows.
DMFIT_FIT = Path("C:/Users/samso/Desktop/CaAlGlass.fxmla")
BRUKER_EXPNO = Path("C:/Users/samso/Desktop/WSU_work/NMR/NMRFAM/DATA/2026-07"
                     "/07062026_SR31648_0Ca-9F-Al_SS_ALP/1903")

assert DMFIT_FIT.exists(), "dmfit example not found -- check the path"
assert BRUKER_EXPNO.exists(), "Bruker EXPNO not found -- check the path"


## Part 1 -- Reproduce `CaAlGlass.fxmla`

`CaAlGlass.fxmla` is a dmfit 1D MAS fit of a Ca-Al glass, `27Al`, MAS = 33296.157 Hz,
Larmor frequency 195.483 MHz. The two dominant lines are both `CzSimple` (dmfit's *simple*,
i.e. non-extended, Czjzek distribution) -- that name maps directly to mrsimulator's plain
`CzjzekDistribution`, not the extended one used in the JCP-paper 25Mg example.

Starting parameters transcribed directly from the saved fit (dmfit units: ppm and kHz):

| site | delta_iso (ppm) | sigma(Cq) (kHz) | relative amplitude |
|---|---|---|---|
| 1 | 66.176 | 4548.65 | 6960.2 |
| 2 | 37.202 | 4406.50 | 1136.1 |

`sCZ_CQ` is dmfit's Czjzek width parameter; `CQ_max` in the file is the derived mode of the
Cq distribution, not an independent input. **Open question to confirm in this phase:** whether
dmfit's `sCZ_CQ` uses the same sigma convention as mrsimulator's `CzjzekDistribution(sigma=...)`,
or needs a scale-factor conversion -- the two programs were not guaranteed to agree on this and
it's exactly the kind of thing a feasibility check exists to catch.


In [ ]:
import mrsimulator
print("mrsimulator", mrsimulator.__version__)

# Discovery step: confirm the exact Czjzek API surface in the installed version
# before assuming a class name from documentation that may have moved.
import mrsimulator.models as models
print([n for n in dir(models) if "zjzek" in n.lower()])


In [ ]:
from mrsimulator import Simulator, SpinSystem, Site
from mrsimulator.spin_system.tensors import SymmetricTensor
from mrsimulator.method.lib import BlochDecayCTSpectrum
from mrsimulator.method import SpectralDimension
from mrsimulator import signal_processor as sp
from mrsimulator.spin_system.isotope import Isotope

MAS_HZ = 33296.15741
LARMOR_27AL_MHZ = 195.483

# B0 that gives 27Al its measured reference frequency, matching the dmfit acquisition.
B0 = Isotope(symbol="27Al").ref_freq_to_B0(reference_frequency=LARMOR_27AL_MHZ * 1e6)
print(f"B0 = {B0:.4f} T")

# TODO once the sigma convention above is confirmed: build the two CzjzekDistribution
# sites, run single_site_system_generator, simulate with BlochDecayCTSpectrum at MAS_HZ,
# and overlay against the experimental trace parsed out of <ExpData> in CaAlGlass.fxmla.


### Fitting

Once the simulated lineshape looks right by eye, hand it to `lmfit` the same way the
PANACEA school material does -- `sf.make_LMFIT_params` + `Minimizer` -- and compare the
refined sigma(Cq)/delta_iso against dmfit's values. Agreement within noise is the actual
pass/fail criterion for this half of Phase 0, not a visual match alone.


In [ ]:
# from mrsimulator.utils import spectral_fitting as sf
# from lmfit import Minimizer
#
# params = sf.make_LMFIT_params(sim, processor)
# minner = Minimizer(sf.LMFIT_min_function, params, fcn_args=(sim, processor, sigma_noise))
# result = minner.minimize()
# print(result.params.pretty_print())


## Part 2 -- Open a real Bruker EXPNO, read-only

EXPNO `1903` -- confirmed from its own `pdata/1/title`: a **19F Hahn echo, 1 rotor period**,
MAS 35.714 kHz, sample SR31648, room temperature, no VT flow. It has both a raw `fid` and an
already-processed `pdata/1/1r` + `1i` -- useful, because TopSpin's own processed spectrum is a
ready-made check for whatever `nmrglue` reads back.


In [ ]:
import numpy as np
import nmrglue as ng

# Read-only: nmrglue.bruker.read only opens acqus + fid, never writes into the EXPNO folder.
dic, fid = ng.bruker.read(str(BRUKER_EXPNO))
print("nucleus (channel 1):", dic["acqus"]["NUC1"])
print("SFO1 (MHz):", dic["acqus"]["SFO1"])
print("MAS rate (Hz):", dic["acqus"].get("MASR"))
print("fid shape:", fid.shape, fid.dtype)


In [ ]:
# Compare against TopSpin's own processing, also read-only.
dic_proc, processed = ng.bruker.read_pdata(str(BRUKER_EXPNO / "pdata" / "1"))
print("processed spectrum shape:", processed.shape)

import matplotlib.pyplot as plt

fid_corrected = ng.bruker.remove_digital_filter(dic, fid)
spectrum = np.fft.fftshift(np.fft.fft(fid_corrected))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(spectrum.real)
axes[0].set_title("nmrglue: FFT of raw fid (uncalibrated)")
axes[1].plot(processed)
axes[1].set_title("TopSpin's own processed spectrum (pdata/1/1r)")
fig.tight_layout()


## Success criteria

- [ ] mrsimulator reproduces `CaAlGlass.fxmla`'s two Czjzek sites within noise, after resolving
      the sigma(Cq) convention question above
- [ ] `nmrglue` reads EXPNO 1903's acqus + fid + processed data correctly, and nothing in the
      EXPNO folder is modified (compare file mtimes before/after)
- [ ] Both checks run from a clean `conda env create -f environment.yml`, no manual patching

If both pass, Phase 1 (the actual Guided/Expert-mode app) is a UI and ingestion-pipeline problem,
not a numerical-feasibility one -- which is the whole point of doing this first.
